# All Baseline Models — PyTorch (Rice Disease Dataset)

**Run all cells top-to-bottom.**  
ImageNet pretrained weights used; only the last block of each model is fine-tuned.  
All results (weights, history, test metrics, profile) saved to `Model/<ModelName>/`.

| Cell | Content |
|------|---------|
| 1 | Imports & Device Setup |
| 2 | Configuration & Data Loading |
| 3 | Shared Utilities |
| 4 | **ResNet50** |
| 5 | **VGG16** |
| 6 | **MobileNetV2** |
| 7 | **InceptionV3** |
| 8 | Final Summary |


In [1]:
import os, copy, json, time, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score, classification_report)

warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type == 'cuda':
    torch.backends.cudnn.benchmark = True

print(f'PyTorch     : {torch.__version__}')
print(f'Torchvision : {torchvision.__version__}')
print(f'Device      : {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU         : {torch.cuda.get_device_name(0)}')
    print(f'CUDA        : {torch.version.cuda}')
    print(f'Total VRAM  : {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB')


PyTorch     : 2.5.1+cu121
Torchvision : 0.20.1+cu121
Device      : cuda
GPU         : NVIDIA RTX 5000 Ada Generation
CUDA        : 12.1
Total VRAM  : 31.99 GB


In [2]:
# ── Hyper-parameters ──────────────────────────────────────────
BATCH_SIZE  = 64
IMAGE_SIZE  = (224, 224)
EPOCHS      = 50
RANDOM_SEED = 42

DATA_DIR   = r'..\Rice_Disease_Dataset'
OUTPUT_DIR = 'Model'
os.makedirs(OUTPUT_DIR, exist_ok=True)

torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print(f'Output directory : {os.path.abspath(OUTPUT_DIR)}')

# ── Discover classes ──────────────────────────────────────────
classes = sorted([
    d for d in os.listdir(DATA_DIR)
    if os.path.isdir(os.path.join(DATA_DIR, d))
])
NUM_CLASSES  = len(classes)
class_to_idx = {cls: i for i, cls in enumerate(classes)}

print('Classes:    ', classes)
print('num_classes:', NUM_CLASSES)

# ── Build filepath / label dataframe ─────────────────────────
filepaths, labels = [], []
for cls in classes:
    cls_dir = os.path.join(DATA_DIR, cls)
    for f in os.listdir(cls_dir):
        if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.gif')):
            filepaths.append(os.path.join(cls_dir, f))
            labels.append(cls)

df = pd.DataFrame({'filepath': filepaths, 'label': labels})
print('Total Images:', len(df))

# ── 80 / 10 / 10 stratified split ────────────────────────────
df_train, df_temp = train_test_split(
    df, test_size=0.20, shuffle=True,
    stratify=df['label'], random_state=123
)
df_val, df_test = train_test_split(
    df_temp, test_size=0.50, shuffle=True,
    stratify=df_temp['label'], random_state=123
)
print('Train:', len(df_train))
print('Val:  ', len(df_val))
print('Test: ', len(df_test))

# ── Class weights ─────────────────────────────────────────────
y_train_encoded  = df_train['label'].map(class_to_idx).values
class_weights    = compute_class_weight(
    class_weight='balanced',
    classes=np.arange(NUM_CLASSES),
    y=y_train_encoded
)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32)

print('Class weights:')
for cls, w in zip(classes, class_weights):
    print(f'  {cls:<25} : {w:.4f}')


Output directory : c:\Users\km1612\Documents\riceMobileNet\RiceMobileNet_final\RiceMobileNEt Code\Experiment_on_Multi_Source_DATASET\Baseline_Transfer_learning\Model
Classes:     ['bacterial_leaf_blight', 'brown_spot', 'healthy', 'leaf_blast', 'leaf_scaled', 'narrow_brown_spot']
num_classes: 6
Total Images: 14758
Train: 11806
Val:   1476
Test:  1476
Class weights:
  bacterial_leaf_blight     : 1.0383
  brown_spot                : 0.8195
  healthy                   : 1.0938
  leaf_blast                : 0.7428
  leaf_scaled               : 0.9760
  narrow_brown_spot         : 1.8811


In [3]:
# =============================================================================
#  SHARED UTILITIES  (run once — used by every model below)
# =============================================================================

# ── Custom Dataset ────────────────────────────────────────────
class ImageDataFrameDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df        = df.reset_index(drop=True)
        self.transform = transform
        self.encoded   = [class_to_idx[l] for l in self.df['label']]
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        img = Image.open(self.df.iloc[idx]['filepath']).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, self.encoded[idx]


# ── Normalization constants ───────────────────────────────────
IMAGENET_NORM = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
MINUSONE_NORM = transforms.Normalize([0.5,   0.5,   0.5  ], [0.5,   0.5,   0.5  ])
IDENTITY_NORM = transforms.Normalize([0.0,   0.0,   0.0  ], [1.0,   1.0,   1.0  ])


# ── Early Stopping ────────────────────────────────────────────
class EarlyStopping:
    def __init__(self, patience=7):
        self.patience     = patience
        self.best_loss    = float('inf')
        self.counter      = 0
        self.best_weights = None
    def __call__(self, val_loss, model):
        if val_loss < self.best_loss:
            self.best_loss    = val_loss
            self.counter      = 0
            self.best_weights = copy.deepcopy(model.state_dict())
            return False
        self.counter += 1
        return self.counter >= self.patience
    def restore(self, model):
        if self.best_weights is not None:
            model.load_state_dict(self.best_weights)


# ── DataLoader factory ────────────────────────────────────────
def make_loaders(train_t, val_t, batch_size=BATCH_SIZE):
    pin = (DEVICE.type == 'cuda')
    train_loader = DataLoader(
        ImageDataFrameDataset(df_train, train_t),
        batch_size=batch_size, shuffle=True,  num_workers=0, pin_memory=pin)
    val_loader = DataLoader(
        ImageDataFrameDataset(df_val, val_t),
        batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=pin)
    test_loader = DataLoader(
        ImageDataFrameDataset(df_test, val_t),
        batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=pin)
    return train_loader, val_loader, test_loader


# ── Keras-style progress bar ──────────────────────────────────
def _bar(current, total, width=30):
    filled = int(width * current / total)
    return '=' * filled + '-' * (width - filled)


# ── Training Engine ───────────────────────────────────────────
def train_model(model, model_name, train_loader, val_loader):
    criterion = nn.CrossEntropyLoss(weight=class_weights_tensor.to(DEVICE), label_smoothing=0.1)
    optimizer = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()), lr=1e-5
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', patience=3, factor=0.5
    )
    scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type == 'cuda'))
    es     = EarlyStopping(patience=7)

    history = {k: [] for k in [
        'loss', 'accuracy', 'precision', 'recall', 'f1',
        'val_loss', 'val_accuracy', 'val_precision', 'val_recall', 'val_f1'
    ]}

    n_batches = len(train_loader)

    print(f'\nModel  : {model_name}')
    print(f'Train batches : {n_batches}  |  Val batches : {len(val_loader)}')
    print(f'Device : {DEVICE}' + (
        f'  ({torch.cuda.get_device_name(0)})' if DEVICE.type == 'cuda' else ''))
    print('-' * 110)

    for epoch in range(1, EPOCHS + 1):
        # ── Train ──
        model.train()
        t0 = time.time()
        ep_loss = 0.0
        ep_preds, ep_labels = [], []

        for bi, (imgs, labels) in enumerate(train_loader, 1):
            imgs   = imgs.to(DEVICE,   non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            with torch.cuda.amp.autocast(enabled=(DEVICE.type == 'cuda')):
                out = model(imgs)
                if isinstance(out, torchvision.models.inception.InceptionOutputs):
                    out = out.logits
                loss = criterion(out, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            ep_loss += loss.item() * imgs.size(0)
            ep_preds.extend(out.argmax(1).detach().cpu().numpy())
            ep_labels.extend(labels.detach().cpu().numpy())

            print(f'\rEpoch {epoch}/{EPOCHS} {bi}/{n_batches} [{_bar(bi, n_batches)}]',
                  end='', flush=True)

        # ── Train metrics ──
        tr_loss = ep_loss / len(train_loader.dataset)
        tr_acc  = accuracy_score(ep_labels, ep_preds)
        tr_prec = precision_score(ep_labels, ep_preds, average='weighted', zero_division=0)
        tr_rec  = recall_score(ep_labels,   ep_preds, average='weighted', zero_division=0)
        tr_f1   = f1_score(ep_labels,       ep_preds, average='weighted', zero_division=0)

        # ── Validate ──
        model.eval()
        vl_loss = 0.0
        vl_preds, vl_labels = [], []

        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs   = imgs.to(DEVICE,   non_blocking=True)
                labels = labels.to(DEVICE, non_blocking=True)
                with torch.cuda.amp.autocast(enabled=(DEVICE.type == 'cuda')):
                    out = model(imgs)
                    if isinstance(out, torchvision.models.inception.InceptionOutputs):
                        out = out.logits
                    loss = criterion(out, labels)
                vl_loss += loss.item() * imgs.size(0)
                vl_preds.extend(out.argmax(1).cpu().numpy())
                vl_labels.extend(labels.cpu().numpy())

        vl_loss /= len(val_loader.dataset)
        vl_acc   = accuracy_score(vl_labels, vl_preds)
        vl_prec  = precision_score(vl_labels, vl_preds, average='weighted', zero_division=0)
        vl_rec   = recall_score(vl_labels,   vl_preds, average='weighted', zero_division=0)
        vl_f1    = f1_score(vl_labels,       vl_preds, average='weighted', zero_division=0)

        elapsed   = time.time() - t0
        step_time = elapsed / n_batches

        print(
            f'\rEpoch {epoch}/{EPOCHS} {n_batches}/{n_batches} [{"=" * 30}] - '
            f'{elapsed:.0f}s {step_time:.1f}s/step - '
            f'loss: {tr_loss:.4f} - accuracy: {tr_acc:.4f} - '
            f'precision: {tr_prec:.4f} - recall: {tr_rec:.4f} - f1: {tr_f1:.4f} - '
            f'val_loss: {vl_loss:.4f} - val_accuracy: {vl_acc:.4f} - '
            f'val_precision: {vl_prec:.4f} - val_recall: {vl_rec:.4f} - val_f1: {vl_f1:.4f}',
            flush=True
        )
        print()

        for key, val in [
            ('loss',          tr_loss), ('accuracy',      tr_acc),
            ('precision',     tr_prec), ('recall',        tr_rec),  ('f1',      tr_f1),
            ('val_loss',      vl_loss), ('val_accuracy',  vl_acc),
            ('val_precision', vl_prec), ('val_recall',    vl_rec),  ('val_f1',  vl_f1),
        ]:
            history[key].append(round(float(val), 6))

        scheduler.step(vl_loss)

        if es(vl_loss, model):
            print(f'  >> Early stopping at epoch {epoch}. '
                  f'Best val_loss = {es.best_loss:.4f}')
            break

    es.restore(model)
    return history


# ── Test Evaluation ───────────────────────────────────────────
def evaluate_on_test(model, test_loader):
    criterion = nn.CrossEntropyLoss(weight=class_weights_tensor.to(DEVICE), label_smoothing=0.1)
    model.eval()
    tl, tp, tg = 0.0, [], []
    with torch.no_grad():
        for imgs, labels in test_loader:
            imgs   = imgs.to(DEVICE,   non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)
            with torch.cuda.amp.autocast(enabled=(DEVICE.type == 'cuda')):
                out = model(imgs)
                if isinstance(out, torchvision.models.inception.InceptionOutputs):
                    out = out.logits
                loss = criterion(out, labels)
            tl += loss.item() * imgs.size(0)
            tp.extend(out.argmax(1).cpu().numpy())
            tg.extend(labels.cpu().numpy())
    tl   /= len(test_loader.dataset)
    acc   = accuracy_score(tg, tp)
    prec  = precision_score(tg, tp, average='weighted', zero_division=0)
    rec   = recall_score(tg,   tp, average='weighted', zero_division=0)
    f1    = f1_score(tg,       tp, average='weighted', zero_division=0)
    print(f'\n{"=" * 60}')
    print('  TEST EVALUATION')
    print(f'{"=" * 60}')
    print(f'  loss      : {tl:.4f}')
    print(f'  accuracy  : {acc:.4f}')
    print(f'  precision : {prec:.4f}')
    print(f'  recall    : {rec:.4f}')
    print(f'  f1_score  : {f1:.4f}')
    print(f'{"=" * 60}')
    print()
    print('Classification Report:')
    print(classification_report(tg, tp, target_names=classes, digits=4))
    return {'test_loss': round(tl,6), 'test_accuracy': round(acc,6),
            'test_precision': round(prec,6), 'test_recall': round(rec,6),
            'test_f1': round(f1,6)}


# ── Model Profiling ───────────────────────────────────────────
def profile_model(model, model_name, model_dir):
    """Compute Total Params, Trainable Params, Disk Size, FLOPs, Inference time, FPS."""
    import time as _time

    # Params
    total_params     = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    # Disk size (from saved .pth)
    pth_path = os.path.join(model_dir, f'{model_name}_weights.pth')
    disk_mb  = os.path.getsize(pth_path) / (1024 ** 2) if os.path.exists(pth_path) else 0.0

    # FLOPs via torch.profiler (fallback: manual count with thop if available)
    dummy = torch.randn(1, 3, *IMAGE_SIZE).to(DEVICE)
    flops_g = 0.0
    try:
        from thop import profile as thop_profile
        macs, _ = thop_profile(model, inputs=(dummy,), verbose=False)
        flops_g = macs * 2 / 1e9   # MACs -> FLOPs (multiply by 2)
    except Exception:
        flops_g = float('nan')

    # Inference time & FPS (100 warm-up + 200 timed runs, batch=1)
    model.eval()
    with torch.no_grad():
        for _ in range(100):
            _ = model(dummy)
    if DEVICE.type == 'cuda':
        torch.cuda.synchronize()
    runs = 200
    t0 = _time.perf_counter()
    with torch.no_grad():
        for _ in range(runs):
            out = model(dummy)
            if isinstance(out, torchvision.models.inception.InceptionOutputs):
                out = out.logits
    if DEVICE.type == 'cuda':
        torch.cuda.synchronize()
    elapsed_s  = _time.perf_counter() - t0
    inf_ms     = (elapsed_s / runs) * 1000
    fps        = runs / elapsed_s

    profile = {
        'model':               model_name,
        'total_params_M':      round(total_params     / 1e6, 3),
        'trainable_params_M':  round(trainable_params / 1e6, 3),
        'disk_size_MB':        round(disk_mb,  2),
        'flops_G':             round(flops_g,  3) if not __import__('math').isnan(flops_g) else 'N/A',
        'inference_ms':        round(inf_ms,   3),
        'fps':                 round(fps,      2),
    }

    print(f'\n  {"─"*55}')
    print(f'  MODEL PROFILE : {model_name}')
    print(f'  {"─"*55}')
    print(f'  Total Params (M)      : {profile["total_params_M"]}')
    print(f'  Trainable Params (M)  : {profile["trainable_params_M"]}')
    print(f'  Disk Size (MB)        : {profile["disk_size_MB"]}')
    print(f'  FLOPs (G)             : {profile["flops_G"]}')
    print(f'  Inference (ms)        : {profile["inference_ms"]}')
    print(f'  FPS                   : {profile["fps"]}')
    print(f'  {"─"*55}\n')
    return profile


# ── Save utilities ────────────────────────────────────────────
def save_model_results(model, model_name, history, test_metrics, profile):
    """Save weights, history, test metrics, and profile into Model/<model_name>/ folder."""
    model_dir = os.path.join(OUTPUT_DIR, model_name)
    os.makedirs(model_dir, exist_ok=True)

    # Weights
    pth_path = os.path.join(model_dir, f'{model_name}_weights.pth')
    torch.save(model.state_dict(), pth_path)
    print(f'  Weights saved  ->  {pth_path}')

    # Training history
    hist_path = os.path.join(model_dir, f'{model_name}_history.json')
    with open(hist_path, 'w') as f:
        json.dump(history, f, indent=2)
    print(f'  History saved  ->  {hist_path}')

    # Test metrics
    test_path = os.path.join(model_dir, f'{model_name}_test_metrics.json')
    with open(test_path, 'w') as f:
        json.dump(test_metrics, f, indent=2)
    print(f'  Test metrics   ->  {test_path}')

    # Profile
    prof_path = os.path.join(model_dir, f'{model_name}_profile.json')
    with open(prof_path, 'w') as f:
        json.dump(profile, f, indent=2)
    print(f'  Profile saved  ->  {prof_path}')

    return model_dir


print('All shared utilities ready.')


All shared utilities ready.


In [4]:
# =============================================================================
#  MODEL 1 : ResNet50
# =============================================================================
print('=' * 70)
print('  MODEL: ResNet50')
print('=' * 70)

# ── Transforms ────────────────────────────────────────────────
resnet_train_t = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.RandomRotation(25),
    transforms.RandomAffine(degrees=0, translate=(0.15, 0.15), shear=5.7),
    transforms.ToTensor(),
    IMAGENET_NORM,
])
resnet_val_t = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.ToTensor(),
    IMAGENET_NORM,
])

resnet_train_loader, resnet_val_loader, resnet_test_loader = make_loaders(resnet_train_t, resnet_val_t)

# ── Architecture ──────────────────────────────────────────────
# weights=IMAGENET1K_V1  → ImageNet pretrained weights
# Freeze all pretrained layers first, then unfreeze layer4 (last residual block)
def build_resnet50(num_classes=NUM_CLASSES):
    model = torchvision.models.resnet50(
        weights=torchvision.models.ResNet50_Weights.IMAGENET1K_V1
    )
    # Freeze all pretrained layers first
    for p in model.parameters():
        p.requires_grad = False

    for p in model.layer4[-1].parameters():
        p.requires_grad = True

    model.fc = nn.Linear(model.fc.in_features, num_classes)

    return model

resnet_model = build_resnet50(NUM_CLASSES).to(DEVICE)
total_p     = sum(p.numel() for p in resnet_model.parameters())
trainable_p = sum(p.numel() for p in resnet_model.parameters() if p.requires_grad)
print(f'Total params     : {total_p:,}')
print(f'Trainable params : {trainable_p:,}')

resnet_history     = train_model(resnet_model, 'ResNet50', resnet_train_loader, resnet_val_loader)
resnet_test_metrics = evaluate_on_test(resnet_model, resnet_test_loader)

resnet_model_dir = save_model_results.__wrapped__ if hasattr(save_model_results, '__wrapped__') else None
resnet_model_dir = os.path.join(OUTPUT_DIR, 'ResNet50')
os.makedirs(resnet_model_dir, exist_ok=True)
torch.save(resnet_model.state_dict(), os.path.join(resnet_model_dir, 'ResNet50_weights.pth'))

resnet_profile = profile_model(resnet_model, 'ResNet50', resnet_model_dir)
save_model_results(resnet_model, 'ResNet50', resnet_history, resnet_test_metrics, resnet_profile)

del resnet_model, resnet_train_loader, resnet_val_loader, resnet_test_loader
if DEVICE.type == 'cuda': torch.cuda.empty_cache()
print('\nResNet50 done.\n')


  MODEL: ResNet50
Total params     : 23,520,326
Trainable params : 4,474,886

Model  : ResNet50
Train batches : 185  |  Val batches : 24
Device : cuda  (NVIDIA RTX 5000 Ada Generation)
--------------------------------------------------------------------------------------------------------------
Epoch 1/50 185/185 [==============================] - 116s 0.6s/step - loss: 1.5461 - accuracy: 0.4984 - precision: 0.5109 - recall: 0.4984 - f1: 0.4917 - val_loss: 1.3323 - val_accuracy: 0.6159 - val_precision: 0.6306 - val_recall: 0.6159 - val_f1: 0.6142

Epoch 2/50 185/185 [==============================] - 29s 0.2s/step - loss: 1.2069 - accuracy: 0.6548 - precision: 0.6702 - recall: 0.6548 - f1: 0.6577 - val_loss: 1.1345 - val_accuracy: 0.6870 - val_precision: 0.7037 - val_recall: 0.6870 - val_f1: 0.6897

Epoch 3/50 185/185 [==============================] - 30s 0.2s/step - loss: 1.0723 - accuracy: 0.7026 - precision: 0.7215 - recall: 0.7026 - f1: 0.7069 - val_loss: 1.0333 - val_accuracy: 0.

In [5]:
# =============================================================================
#  MODEL 2 : VGG16
# =============================================================================
print('=' * 70)
print('  MODEL: VGG16')
print('=' * 70)

# ── Transforms ────────────────────────────────────────────────
vgg_train_t = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.RandomRotation(25),
    transforms.RandomAffine(degrees=0, translate=(0.15, 0.15), shear=5.7),
    transforms.ToTensor(),
    IMAGENET_NORM,
])
vgg_val_t = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.ToTensor(),
    IMAGENET_NORM,
])

vgg_train_loader, vgg_val_loader, vgg_test_loader = make_loaders(vgg_train_t, vgg_val_t)

# ── Architecture ──────────────────────────────────────────────
# weights=IMAGENET1K_V1  → ImageNet pretrained weights
# VGG16 has no named 'layerN' blocks; last conv block = features[24:31] (Conv block 5)
# Freeze all pretrained layers first, then unfreeze last conv block (features[24:])
def build_vgg16(num_classes=NUM_CLASSES):
    model = torchvision.models.vgg16(weights=torchvision.models.VGG16_Weights.IMAGENET1K_V1)
    # Freeze all pretrained layers first
    for p in model.parameters():
        p.requires_grad = False
    # Fine-tune last block (conv block 5: features indices 24-30)
    for p in model.features[25:].parameters():
        p.requires_grad = True
    # Replace classifier
    model.avgpool    = nn.AdaptiveAvgPool2d(1)
    model.classifier = nn.Sequential(
        nn.Flatten(),
        nn.Linear(512, 512),
        nn.ReLU(inplace=True),
        nn.Dropout(0.5),
        nn.Linear(512, num_classes),
    )
    return model

vgg_model = build_vgg16(NUM_CLASSES).to(DEVICE)
total_p     = sum(p.numel() for p in vgg_model.parameters())
trainable_p = sum(p.numel() for p in vgg_model.parameters() if p.requires_grad)
print(f'Total params     : {total_p:,}')
print(f'Trainable params : {trainable_p:,}')

vgg_history      = train_model(vgg_model, 'VGG16', vgg_train_loader, vgg_val_loader)
vgg_test_metrics = evaluate_on_test(vgg_model, vgg_test_loader)

vgg_model_dir = os.path.join(OUTPUT_DIR, 'VGG16')
os.makedirs(vgg_model_dir, exist_ok=True)
torch.save(vgg_model.state_dict(), os.path.join(vgg_model_dir, 'VGG16_weights.pth'))

vgg_profile = profile_model(vgg_model, 'VGG16', vgg_model_dir)
save_model_results(vgg_model, 'VGG16', vgg_history, vgg_test_metrics, vgg_profile)

del vgg_model, vgg_train_loader, vgg_val_loader, vgg_test_loader
if DEVICE.type == 'cuda': torch.cuda.empty_cache()
print('\nVGG16 done.\n')


  MODEL: VGG16
Total params     : 14,980,422
Trainable params : 4,985,350

Model  : VGG16
Train batches : 185  |  Val batches : 24
Device : cuda  (NVIDIA RTX 5000 Ada Generation)
--------------------------------------------------------------------------------------------------------------
Epoch 1/50 185/185 [==============================] - 30s 0.2s/step - loss: 1.7456 - accuracy: 0.3010 - precision: 0.3839 - recall: 0.3010 - f1: 0.2927 - val_loss: 1.6057 - val_accuracy: 0.4221 - val_precision: 0.4849 - val_recall: 0.4221 - val_f1: 0.3819

Epoch 2/50 185/185 [==============================] - 29s 0.2s/step - loss: 1.4640 - accuracy: 0.4932 - precision: 0.5090 - recall: 0.4932 - f1: 0.4904 - val_loss: 1.3654 - val_accuracy: 0.4864 - val_precision: 0.6093 - val_recall: 0.4864 - val_f1: 0.4619

Epoch 3/50 185/185 [==============================] - 30s 0.2s/step - loss: 1.2980 - accuracy: 0.5680 - precision: 0.5813 - recall: 0.5680 - f1: 0.5674 - val_loss: 1.2677 - val_accuracy: 0.5420 - 

In [6]:
# =============================================================================
#  MODEL 3 : MobileNetV2
# =============================================================================
print('=' * 70)
print('  MODEL: MobileNetV2')
print('=' * 70)

# ── Transforms ────────────────────────────────────────────────
mobilenet_train_t = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.RandomRotation(25),
    transforms.RandomAffine(degrees=0, translate=(0.15, 0.15), shear=5.7),
    transforms.ToTensor(),
    MINUSONE_NORM,
])
mobilenet_val_t = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.ToTensor(),
    MINUSONE_NORM,
])

mobilenet_train_loader, mobilenet_val_loader, mobilenet_test_loader = make_loaders(
    mobilenet_train_t, mobilenet_val_t
)

# ── Architecture ──────────────────────────────────────────────
# weights=IMAGENET1K_V1  → ImageNet pretrained weights
# MobileNetV2: features is a Sequential of 19 InvertedResidual / Conv blocks
# Last block = features[18] (the final point-wise conv before pooling)
# Freeze all pretrained layers first, then unfreeze features[18] (last block)
def build_mobilenetv2(num_classes=NUM_CLASSES):
    model = torchvision.models.mobilenet_v2(weights=torchvision.models.MobileNet_V2_Weights.IMAGENET1K_V1)
    # Freeze all pretrained layers first
    for p in model.parameters():
        p.requires_grad = False
    # Fine-tune last block (features[18])
    for p in model.features[18].parameters():
        p.requires_grad = True
    # Replace classifier
    model.classifier = nn.Sequential(
        nn.Dropout(0.3),
        nn.Linear(1280, num_classes),
    )
    return model

mobilenet_model = build_mobilenetv2(NUM_CLASSES).to(DEVICE)
total_p     = sum(p.numel() for p in mobilenet_model.parameters())
trainable_p = sum(p.numel() for p in mobilenet_model.parameters() if p.requires_grad)
print(f'Total params     : {total_p:,}')
print(f'Trainable params : {trainable_p:,}')

mobilenet_history      = train_model(mobilenet_model, 'MobileNetV2', mobilenet_train_loader, mobilenet_val_loader)
mobilenet_test_metrics = evaluate_on_test(mobilenet_model, mobilenet_test_loader)

mobilenet_model_dir = os.path.join(OUTPUT_DIR, 'MobileNetV2')
os.makedirs(mobilenet_model_dir, exist_ok=True)
torch.save(mobilenet_model.state_dict(), os.path.join(mobilenet_model_dir, 'MobileNetV2_weights.pth'))

mobilenet_profile = profile_model(mobilenet_model, 'MobileNetV2', mobilenet_model_dir)
save_model_results(mobilenet_model, 'MobileNetV2', mobilenet_history, mobilenet_test_metrics, mobilenet_profile)

del mobilenet_model, mobilenet_train_loader, mobilenet_val_loader, mobilenet_test_loader
if DEVICE.type == 'cuda': torch.cuda.empty_cache()
print('\nMobileNetV2 done.\n')


  MODEL: MobileNetV2
Total params     : 2,231,558
Trainable params : 419,846

Model  : MobileNetV2
Train batches : 185  |  Val batches : 24
Device : cuda  (NVIDIA RTX 5000 Ada Generation)
--------------------------------------------------------------------------------------------------------------
Epoch 1/50 185/185 [==============================] - 39s 0.2s/step - loss: 1.7438 - accuracy: 0.2869 - precision: 0.3228 - recall: 0.2869 - f1: 0.2784 - val_loss: 1.6375 - val_accuracy: 0.4173 - val_precision: 0.4970 - val_recall: 0.4173 - val_f1: 0.3962

Epoch 2/50 185/185 [==============================] - 32s 0.2s/step - loss: 1.5580 - accuracy: 0.4495 - precision: 0.4720 - recall: 0.4495 - f1: 0.4479 - val_loss: 1.5139 - val_accuracy: 0.4885 - val_precision: 0.5751 - val_recall: 0.4885 - val_f1: 0.4823

Epoch 3/50 185/185 [==============================] - 32s 0.2s/step - loss: 1.4512 - accuracy: 0.5185 - precision: 0.5445 - recall: 0.5185 - f1: 0.5205 - val_loss: 1.4282 - val_accuracy: 

In [11]:
# =============================================================================
#  MODEL 4 : InceptionV3
# =============================================================================
print('=' * 70)
print('  MODEL: InceptionV3')
print('=' * 70)

INCEPTION_IMAGE_SIZE = (224, 224)

# ── Transforms ────────────────────────────────────────────────
inception_train_t = transforms.Compose([
    transforms.Resize(INCEPTION_IMAGE_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(25),
    transforms.RandomAffine(
        degrees=0,
        translate=(0.2, 0.2),
        scale=(0.7, 1.3),
        shear=8.5,
    ),
    transforms.ColorJitter(brightness=0.3),
    transforms.ToTensor(),
    MINUSONE_NORM,
])
inception_val_t = transforms.Compose([
    transforms.Resize(INCEPTION_IMAGE_SIZE),
    transforms.ToTensor(),
    MINUSONE_NORM,
])

inception_train_loader, inception_val_loader, inception_test_loader = make_loaders(
    inception_train_t, inception_val_t
)

# ── Architecture ──────────────────────────────────────────────
# weights=IMAGENET1K_V1  → ImageNet pretrained weights
# aux_logits=False: disable auxiliary branch (not needed for fine-tuning)
# InceptionV3 last block: Mixed_7c (the last Inception module before avg pool)
# Freeze all pretrained layers first, then unfreeze Mixed_7c (last block)
def build_inceptionv3(num_classes=NUM_CLASSES):
    model = torchvision.models.inception_v3(
        weights=torchvision.models.Inception_V3_Weights.IMAGENET1K_V1,
        aux_logits=True
    )

    # Very important for 224x224 input
    model.aux_logits = False
    model.AuxLogits = None

    # Freeze all pretrained layers
    for p in model.parameters():
        p.requires_grad = False

    # Fine-tune last Inception block
    for p in model.Mixed_7c.parameters():
        p.requires_grad = True

    # Replace classifier
    model.fc = nn.Linear(model.fc.in_features, num_classes)

    return model

inception_model = build_inceptionv3(NUM_CLASSES).to(DEVICE)
total_p     = sum(p.numel() for p in inception_model.parameters())
trainable_p = sum(p.numel() for p in inception_model.parameters() if p.requires_grad)
print(f'Total params     : {total_p:,}')
print(f'Trainable params : {trainable_p:,}')

inception_history      = train_model(inception_model, 'InceptionV3', inception_train_loader, inception_val_loader)
inception_test_metrics = evaluate_on_test(inception_model, inception_test_loader)

inception_model_dir = os.path.join(OUTPUT_DIR, 'InceptionV3')
os.makedirs(inception_model_dir, exist_ok=True)
torch.save(inception_model.state_dict(), os.path.join(inception_model_dir, 'InceptionV3_weights.pth'))

inception_profile = profile_model(inception_model, 'InceptionV3', inception_model_dir)
save_model_results(inception_model, 'InceptionV3', inception_history, inception_test_metrics, inception_profile)

del inception_model, inception_train_loader, inception_val_loader, inception_test_loader
if DEVICE.type == 'cuda': torch.cuda.empty_cache()
print('\nInceptionV3 done.\n')


  MODEL: InceptionV3
Total params     : 21,797,862
Trainable params : 6,089,094

Model  : InceptionV3
Train batches : 185  |  Val batches : 24
Device : cuda  (NVIDIA RTX 5000 Ada Generation)
--------------------------------------------------------------------------------------------------------------
Epoch 1/50 185/185 [==============================] - 43s 0.2s/step - loss: 1.7290 - accuracy: 0.3101 - precision: 0.3389 - recall: 0.3101 - f1: 0.3116 - val_loss: 1.5862 - val_accuracy: 0.4709 - val_precision: 0.5429 - val_recall: 0.4709 - val_f1: 0.4552

Epoch 2/50 185/185 [==============================] - 39s 0.2s/step - loss: 1.5311 - accuracy: 0.4766 - precision: 0.4957 - recall: 0.4766 - f1: 0.4779 - val_loss: 1.4277 - val_accuracy: 0.5237 - val_precision: 0.5886 - val_recall: 0.5237 - val_f1: 0.5145

Epoch 3/50 185/185 [==============================] - 41s 0.2s/step - loss: 1.4131 - accuracy: 0.5314 - precision: 0.5474 - recall: 0.5314 - f1: 0.5336 - val_loss: 1.3384 - val_accurac

In [12]:
# =============================================================================
#  TRAINING COMPLETE — all results saved under Model/<ModelName>/
# =============================================================================
import glob as _glob

print('=' * 70)
print('  ALL MODELS TRAINING COMPLETE')
print('=' * 70)
print()
print('Saved files:')
for p in sorted(_glob.glob(os.path.join(OUTPUT_DIR, '**', '*'), recursive=True)):
    if os.path.isfile(p):
        size_kb = os.path.getsize(p) / 1024
        print(f'  {p:<65}  ({size_kb:.1f} KB)')

print()
print('-- How to reload weights (example: ResNet50) --')
print("  model = build_resnet50(NUM_CLASSES)")
print("  model.load_state_dict(torch.load('Model/ResNet50/ResNet50_weights.pth', map_location=DEVICE))")
print("  model.to(DEVICE).eval()")
print()
print('-- How to reload history (example: ResNet50) --')
print("  with open('Model/ResNet50/ResNet50_history.json') as f:")
print("      h = json.load(f)")
print("  # h['val_accuracy'], h['val_f1'], h['loss'], ...")


  ALL MODELS TRAINING COMPLETE

Saved files:
  Model\InceptionV3\InceptionV3_history.json                         (7.5 KB)
  Model\InceptionV3\InceptionV3_profile.json                         (0.2 KB)
  Model\InceptionV3\InceptionV3_test_metrics.json                    (0.1 KB)
  Model\InceptionV3\InceptionV3_weights.pth                          (85480.6 KB)
  Model\MobileNetV2\MobileNetV2_history.json                         (7.5 KB)
  Model\MobileNetV2\MobileNetV2_profile.json                         (0.2 KB)
  Model\MobileNetV2\MobileNetV2_test_metrics.json                    (0.1 KB)
  Model\MobileNetV2\MobileNetV2_weights.pth                          (8959.3 KB)
  Model\ResNet50\ResNet50_history.json                               (7.5 KB)
  Model\ResNet50\ResNet50_profile.json                               (0.2 KB)
  Model\ResNet50\ResNet50_test_metrics.json                          (0.1 KB)
  Model\ResNet50\ResNet50_weights.pth                                (92189.3 KB)
  Model\